0. 준비

In [2]:
import pandas as pd

# 캐글 여행정보 데이터
kaggle = pd.read_csv('../data/raw/travel_destinations.csv')
# 외교부 안전정보 데이터
safety = pd.read_json('../data/raw/safety_info.json')
# 외교부 여행경보 데이터
alarm = pd.read_json('../data/raw/alarm.json')

# 1) 결측치
print('캐글 결측치:', kaggle.isna().sum())
print('외교부 안전정보 결측치:', safety.isna().sum())

# 2) 공백 제거
kaggle['Country'] = kaggle['Country'].str.strip()
safety['영문국가명'] = safety['영문국가명'].str.strip()

# 3) 중복
print('외교부 데이터 영문국가명 중복:',safety['영문국가명'].duplicated().sum())

캐글 결측치: City                   0
Country                0
Category               0
Best_Time_to_Travel    0
dtype: int64
외교부 안전정보 결측치: 국가명         0
영문국가명       0
id          0
제목          0
내용        666
작성일         0
첨부파일     5472
dtype: int64
외교부 데이터 영문국가명 중복: 5792


중복 5792 -> 전체 데이터 약 6000건 중 나라가 얼마 없어서 한 나라당 게시글 여러개가 중복됨

1. 캐글 국가 뽑기

In [3]:

countries = kaggle['Country'].drop_duplicates().to_frame()
print(countries)
display(countries.head())
print(len(countries))

                  Country
0          United Kingdom
1                  France
2                   Italy
3                 Germany
4                   Spain
5          Czech Republic
6             Netherlands
7                  Greece
8                Portugal
9             Switzerland
10                Iceland
11                Austria
12                Ireland
13                Denmark
14                 Sweden
15                  Japan
17            South Korea
18                  China
21               Thailand
23              Singapore
24              Indonesia
25                Vietnam
27                  India
30   United Arab Emirates
31                 Turkey
32                    USA
39                 Canada
42                 Mexico
44                 Brazil
45              Argentina
46                   Peru
48                  Chile
49               Colombia
50              Australia
53            New Zealand
65                 Poland
66                Hungary
67          

,Country
0,United Kingdom
1,France
2,Italy
3,Germany
4,Spain


44


2. 외교부 국가명 뽑기

In [4]:
countries_kor = safety[['국가명','영문국가명']].drop_duplicates()
print(len(countries_kor))
display(countries_kor.head())

165


,국가명,영문국가명
0,베네수엘라,Venezuela
1,방글라데시,Bangladesh
2,파라과이,Paraguay
3,중국,China
5,레바논,Lebanon


3. 영문명으로 붙이기

In [5]:
countries_kor = countries_kor.rename(columns={'영문국가명':'Country'})
merged = pd.merge(countries, countries_kor, on='Country', how='left')
display(merged.head(20))

,Country,국가명
0,United Kingdom,영국
1,France,프랑스
2,Italy,이탈리아
3,Germany,독일
4,Spain,스페인
5,Czech Republic,체코
6,Netherlands,네덜란드
7,Greece,그리스
8,Portugal,포르투갈
9,Switzerland,스위스


4. ISO 코드 붙이기

In [6]:
import pycountry

def get_iso(name):
    try:
        c = pycountry.countries.search_fuzzy(name)[0]
        return c.alpha_2, c.alpha_3
    except:
        return None, None

merged[["iso2", "iso3"]] = merged["Country"].apply(
    lambda x: pd.Series(get_iso(x))
)
merged[merged["iso3"].isna()]

,Country,국가명,iso2,iso3
24,Turkey,NaN,NaN,NaN


5. 안 붙은 것만 보기

In [7]:
missing = merged[merged['국가명'].isna()]
print(len(missing))
missing['Country'].tolist()

3


['South Korea', 'Turkey', 'USA']

5-1. 미국 국가명 결측치 처리

In [8]:
merged.loc[merged['Country'] == 'USA', '국가명'] = '미국'

5-2. Turkey ISO 채우기 (국가명 변경으로 인한 결측치)

In [9]:
merged.loc[merged['Country'] == 'Turkey', ['iso2', 'iso3']] = ['TR', 'TUR']
merged.loc[merged['Country'] == 'Turkey', ['국가명', 'iso2', 'iso3']] = ['튀르키예', 'TR', 'TUR']

5-3. South Korea를 제외한 나머지 파일로 저장

In [15]:
merged = merged[merged['Country'] != 'South Korea']
merged = merged.rename(columns={'Country': 'kaggle_country', '국가명': 'country_kr'})
# merged.to_csv('../data/processed/country_master.csv', index=False, encoding='utf-8-sig')

KeyError: 'Country'

## 캐글 데이터 국가 고유값 확인

In [ ]:
print(kaggle['Country'].unique())
# 캐글 나라 고유값 총 44개

<StringArray>
[      'United Kingdom',               'France',                'Italy',
              'Germany',                'Spain',       'Czech Republic',
          'Netherlands',               'Greece',             'Portugal',
          'Switzerland',              'Iceland',              'Austria',
              'Ireland',              'Denmark',               'Sweden',
                'Japan',          'South Korea',                'China',
             'Thailand',            'Singapore',            'Indonesia',
              'Vietnam',                'India', 'United Arab Emirates',
               'Turkey',                  'USA',               'Canada',
               'Mexico',               'Brazil',            'Argentina',
                 'Peru',                'Chile',             'Colombia',
            'Australia',          'New Zealand',               'Poland',
              'Hungary',              'Belgium',               'Norway',
             'Malaysia',             

## 캐글 & 외교부(safety) 국가 교집합 확인

In [ ]:
kaggle_list = kaggle['Country'].unique()
safety_list = safety['영문국가명'].unique()

intst = [i for i in kaggle_list if i in safety_list]
print(len(intst))

41


## 캐글 & 외교부(alarm) 국가 교집합 확인

In [ ]:
alarm_list = alarm['영문국가명'].unique()

intst = [i for i in kaggle_list if i in alarm_list]
print(len(intst))

26


외교부(alarm) 데이터 결측치 확인

In [ ]:
# 1) 결측치
print(f'[캐글 결측치] 전체 {len(kaggle)}건\n', kaggle.isna().sum())
print(f'\n[외교부 안전정보 결측치] 전체 {len(safety)}건\n', safety.isna().sum())
print(f'\n[외교부 여행경보 결측치] 전체 {len(alarm)}건\n', alarm.isna().sum())

# 2) 공백 제거
kaggle['Country'] = kaggle['Country'].str.strip()
safety['영문국가명'] = safety['영문국가명'].str.strip()

# 3) 중복
print('\n외교부 데이터 영문국가명 중복:',safety['영문국가명'].duplicated().sum())

[캐글 결측치] 전체 111건
 City                   0
Country                0
Category               0
Best_Time_to_Travel    0
dtype: int64

[외교부 안전정보 결측치] 전체 5957건
 국가명         0
영문국가명       0
id          0
제목          0
내용        666
작성일         0
첨부파일     5472
dtype: int64

[외교부 여행경보 결측치] 전체 208건
 국가명         0
영문국가명       0
ISO 코드      0
대륙 코드       0
영문 대륙명      0
한글 대륙명      0
경보단계        0
경보내용        0
작성일       208
dtype: int64

외교부 데이터 영문국가명 중복: 5792
